In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
ROOT = Path("/home/mohamed/SDD/hackathon/sia-predicting-short-form-video-popularity")
N_SPLITS = 5
SEED = 42

import pandas as pd

train = pd.read_csv(ROOT / "train_full_merged.csv")
test  = pd.read_csv(ROOT / "test_full_merged.csv")
y_df  = pd.read_csv(ROOT / "y_train.csv", sep=";")

# Harmoniser ID
def normalize_id(s):
    s = s.astype(str)
    for pat in [r"^VIDEO_", r"^video_", r"^Video_"]:
        s = s.str.replace(pat, "", regex=True)
    return s.str.strip()  # important

train["ID"] = normalize_id(train["ID"])
test["ID"]  = normalize_id(test["ID"])

# y_df: s'assurer que la colonne ID est bien nommée
if "ID" not in y_df.columns:
    y_df = y_df.rename(columns={y_df.columns[0]: "ID"})
y_df["ID"] = normalize_id(y_df["ID"])

# ✅ IMPORTANT: enlever popularity existant dans train pour éviter tout conflit
if "popularity" in train.columns:
    train = train.drop(columns=["popularity"])

# (optionnel mais recommandé) vérifier unicité des IDs dans y_df
if y_df["ID"].duplicated().any():
    dups = y_df.loc[y_df["ID"].duplicated(keep=False), "ID"].value_counts().head(10)
    raise ValueError(f"IDs dupliqués dans y_df (top 10):\n{dups}")

# Merge target (sans popularity_x/y)
train = train.merge(y_df[["ID", "popularity"]], on="ID", how="left", validate="m:1")

print(train[["ID", "popularity"]].head(5))
print(f"Train shape: {train.shape} | Target nulls: {train['popularity'].isna().sum()}")
print(f"Test shape : {test.shape}")
print(train[["ID", "popularity"]].head(5))
print(f"Train shape: {train.shape} | Target nulls: {train['popularity'].isna().sum()}")
print(f"Test shape : {test.shape}")

# =========================
# 2) Préparer features
# =========================
drop_cols = ["ID", "popularity"]

# Colonnes non numériques → drop (LightGBM veut du numérique ou category)
non_numeric = train.drop(columns=drop_cols, errors="ignore").select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nColonnes non-numériques droppées ({len(non_numeric)}): {non_numeric[:10]}{'...' if len(non_numeric)>10 else ''}")

feature_cols = [c for c in train.columns if c not in drop_cols + non_numeric]

X = train[feature_cols].copy()
y = train["popularity"].copy()
X_test_final = test[feature_cols].copy()

print(f"\nNombre de features: {len(feature_cols)}")
print(f"NaN dans X: {X.isna().sum().sum()} | NaN dans X_test: {X_test_final.isna().sum().sum()}")

# =========================
# 3) LightGBM params
# =========================
lgb_params = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    0.03,
    "num_leaves":       127,
    "max_depth":        -1,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "n_jobs":           -1,
    "verbose":          -1,
    "random_state":     SEED,
}

# =========================
# 4) Cross-validation + OOF
# =========================
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_preds   = np.zeros(len(X))
test_preds  = np.zeros(len(X_test_final))
feature_imp = np.zeros(len(feature_cols))

print(f"\n{'='*50}")
print(f"KFold CV — {N_SPLITS} folds")
print(f"{'='*50}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(n_estimators=3000, **lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds         += model.predict(X_test_final) / N_SPLITS
    feature_imp        += model.feature_importances_ / N_SPLITS

    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"  Fold {fold+1} | Best iter: {model.best_iteration_:4d} | RMSE: {fold_rmse:.4f}")

oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\n{'='*50}")
print(f"OOF RMSE global : {oof_rmse:.4f}")
print(f"{'='*50}")

# =========================
# 5) Feature importance top 20
# =========================
fi_df = pd.DataFrame({"feature": feature_cols, "importance": feature_imp})
fi_df = fi_df.sort_values("importance", ascending=False).head(20)
print("\nTop 20 features:")
print(fi_df.to_string(index=False))

# =========================
# 6) Submission
# =========================
submission = pd.DataFrame({
    "ID":         test["ID"].values,
    "popularity": test_preds
})

# Vérif
assert submission["ID"].isna().sum() == 0, "IDs manquants dans la submission!"
assert submission["popularity"].isna().sum() == 0, "Prédictions NaN dans la submission!"

out_path = ROOT / "submission_lgbm.csv"
submission.to_csv(out_path, index=False)

print(f"\n✅ Submission sauvegardée : {out_path}")
print(f"   Shape : {submission.shape}")
print(f"\nAperçu:")
print(submission.head(10).to_string(index=False))
print(f"\nStats popularity prédit:")
print(submission["popularity"].describe().round(4))

                    ID  popularity
0  7602656035161050390    8.603777
1  7590718903144287510    8.537734
2  7571821778746592534   11.208495
3  7569927329154190614   11.982352
4  7566270741134462230    8.678487
Train shape: (1348, 103) | Target nulls: 0
Test shape : (338, 102)
                    ID  popularity
0  7602656035161050390    8.603777
1  7590718903144287510    8.537734
2  7571821778746592534   11.208495
3  7569927329154190614   11.982352
4  7566270741134462230    8.678487
Train shape: (1348, 103) | Target nulls: 0
Test shape : (338, 102)

Colonnes non-numériques droppées (5): ['album', 'artist', 'channel', 'track', 'uploader']

Nombre de features: 96
NaN dans X: 66 | NaN dans X_test: 9

KFold CV — 5 folds
[200]	valid_0's rmse: 1.2777
  Fold 1 | Best iter:  205 | RMSE: 1.2760
[200]	valid_0's rmse: 1.25788
  Fold 2 | Best iter:  176 | RMSE: 1.2553
[200]	valid_0's rmse: 1.32808
  Fold 3 | Best iter:  150 | RMSE: 1.3226
[200]	valid_0's rmse: 1.34856
  Fold 4 | Best iter:  166 | R

In [15]:
# % de NaN par feature (train/test)
nan_train = X.isna().mean().sort_values(ascending=False)
nan_test  = X_test_final.isna().mean().sort_values(ascending=False)

print("Top 30 colonnes les plus NaN (train):")
print(nan_train.head(30))

print("\nColonnes très NaN (>=50% train):", (nan_train >= 0.50).sum())

Top 30 colonnes les plus NaN (train):
lum_f3_sharpness              0.01632
lum_f3_brightness             0.01632
lum_f3_saturation             0.01632
aspect_ratio                  0.00000
nb_hashtags                   0.00000
video_duration                0.00000
release_year                  0.00000
text_len                      0.00000
emoji_density                 0.00000
feat_meteo                    0.00000
feat_cta                      0.00000
feat_action                   0.00000
lang_lang_detected_de         0.00000
nb_emojis                     0.00000
nb_mentions                   0.00000
hashtag_density               0.00000
lang_lang_detected_it         0.00000
lang_lang_detected_unknown    0.00000
desc_len                      0.00000
num_words                     0.00000
num_hashtags                  0.00000
num_mentions                  0.00000
num_emojis                    0.00000
hashtag_ratio                 0.00000
width                         0.00000
height      